## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. Download the dataset first: click [Weight vs Age of chicks on different diets.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/W4%20Gen%20AI/W4D3/Weight%20vs%20Age%20of%20chicks%20on%20different%20diets.zip). It downloads `Weight vs Age of chicks on different diets.zip`. Save it in `C:\Users\sivan\Datasets`.
3. When the upload cell asks for a file, choose `Weight vs Age of chicks on different diets.zip` from `C:\Users\sivan\Datasets`.

**Run the cells from top to bottom** (Shift+Enter runs one cell). After a restart, run them again from the top.

# 👩‍🏫 👩🏿‍🏫 What You’ll learn
- Conducting detailed statistical analyses to understand the impact of variables on a given outcome.
- Data exploration and hypothesis testing.
- Advanced data analysis techniques applicable in biological and health sciences.

# 🛠️ What you will create
- A thorough statistical analysis of a dataset related to diet and growth, providing insights into the effects of different diets on the growth of chicks.

---
# Exercise: Analysis of Diet Effects on Growth
- **Objective:** Conduct a detailed statistical analysis to understand how different diets impact the growth of chicks.
- **Dataset:** The dataset consists of information about the weight and age of chicks on different diets. It is available on Kaggle: [Weight vs Age of Chicks on Different Diets](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/W4%20Gen%20AI/W4D3/Weight%20vs%20Age%20of%20chicks%20on%20different%20diets.zip)
- **Tasks to Perform:**
  - **Data Exploration:** Examine the dataset to understand its structure and contents. Identify key variables such as age, weight, and diet type.
  - **Data Visualization:** Create visualizations to show how weight varies with age for chicks on different diets.
  - **Statistical Testing:** Use appropriate statistical tests (e.g., ANOVA) to determine if the differences in weight gain among the different diets are statistically significant.
  - **Growth Analysis:** Analyze the growth rate of chicks under different dietary conditions.
  - **Report Findings:** Compile your findings into a comprehensive report that includes both the statistical analysis and practical implications.
- **Tools and Techniques:**
  - Use Python libraries like Pandas for data manipulation.
  - Employ Matplotlib or Seaborn for visualization.
  - Apply statistical methods using SciPy.
- **Expected Outcome:**
  - A detailed report that provides insights into the effectiveness of different diets on the growth of chicks, supported by statistical evidence.

## Setup
Run this cell first. It imports the libraries used in the analysis.

In [ ]:
import pandas as pd
# pandas: for working with tables (DataFrames)
import numpy as np
# numpy: for math on whole columns
import matplotlib.pyplot as plt
# matplotlib: for drawing charts
import seaborn as sns
# seaborn: for nicer statistical charts (built on matplotlib)
from scipy import stats
# stats: the statistics part of SciPy (f_oneway, ttest_ind, linregress ...)

---
# Task 1: Data Exploration

In [ ]:
from google.colab import files
# the Colab tool for uploading files from your computer
uploaded = files.upload()
# choose "Weight vs Age of chicks on different diets.zip" (saved in a variable so its contents are not printed)
!unzip "Weight vs Age of chicks on different diets.zip"
# unzip it: you get the file "ChickWeight.csv"

In [ ]:
chicks = pd.read_csv('ChickWeight.csv')
# load the CSV file into a DataFrame called chicks
print(chicks.shape)
# (number of rows, number of columns)
print(chicks.head(13))
# the first 13 rows: all the measurements of chick 1

In [ ]:
chicks.info()
# the column names, their types and how many values are not empty

**What the table contains:** 578 rows (measurements) and 5 columns, with no empty cells:
- **Unnamed: 0**: just a row number saved by mistake when the file was written, so we drop it.
- **weight**: the chick's weight in grams (**the result we study**).
- **Time**: the chick's age in days on the weighing day (0 = birth, up to 21).
- **Chick**: the chick's ID number (1–50). Each chick appears many times, once per weighing.
- **Diet**: the diet group, 1 to 4 (**the cause we test**). It is stored as a number but it is really a label.

In [ ]:
chicks = chicks.drop(columns=['Unnamed: 0'])
# remove the extra row-number column saved in the file
chicks = chicks.rename(columns={'weight': 'Weight', 'Time': 'Age'})
# clearer names: Weight (grams) and Age (days since birth)
chicks['Diet'] = chicks['Diet'].astype('category')
# Diet is a group label (1-4), not an amount, so we store it as a category
print(chicks.dtypes)
# check the new types
print(chicks.describe())
# basic statistics of the number columns

In [ ]:
print('Missing values:')
# title of the next print
print(chicks.isnull().sum())
# count empty cells in each column
print('Duplicate rows:', chicks.duplicated().sum())
# rows that are exact copies
print('Chicks per diet:')
# title of the next print
print(chicks.groupby('Diet', observed=True)['Chick'].nunique())
# the number of different chicks in each diet
print('Measurement days:', sorted(chicks['Age'].unique().tolist()))
# the days on which the chicks were weighed

In [ ]:
per_chick = chicks.groupby('Chick').agg(diet=('Diet', 'first'), measurements=('Age', 'size'), last_age=('Age', 'max'))
# for every chick: its diet, how many times it was weighed, and the last day it was weighed
print(per_chick[per_chick['last_age'] < 21])
# the chicks that were not weighed until day 21 (they left the study early)

**What we learn:**
- **50 chicks**: 20 on Diet 1 and 10 on each of Diets 2, 3 and 4. Diet 1 has twice as many chicks, which is why we compare averages rather than totals.
- Every chick was planned to be weighed **12 times** (days 0, 2, 4, …, 20 and 21). There are no empty cells, but **some rows are simply missing**: **5 chicks** left the study early (probably died). Chick 18 has only 2 weighings, chick 16 stops at day 12, chick 15 at day 14, chick 44 at day 18 and chick 8 at day 20. **Four of the five are on Diet 1.**
- This matters: the late-day averages of Diet 1 are based on fewer chicks, and the chicks that dropped out were mostly light ones (see the short flat lines at the bottom of the next chart).

---
# Task 2: Data Visualization

In [ ]:
plt.figure(figsize=(10, 5))
# a new chart
sns.lineplot(data=chicks, x='Age', y='Weight', hue='Diet', units='Chick', estimator=None, alpha=0.5)
# one thin line per chick (units='Chick', estimator=None), colored by diet
plt.title('Weight of every chick by age')
# chart title
plt.xlabel('Age (days)')
# label of the x axis
plt.ylabel('Weight (grams)')
# label of the y axis
plt.show()
# show the chart

In [ ]:
plt.figure(figsize=(10, 5))
# a new chart
sns.lineplot(data=chicks, x='Age', y='Weight', hue='Diet', marker='o')
# the average weight per diet at each age; the shaded band is the 95% confidence interval
plt.title('Average weight by age for each diet')
# chart title
plt.xlabel('Age (days)')
# label of the x axis
plt.ylabel('Average weight (grams)')
# label of the y axis
plt.show()
# show the chart

In [ ]:
mean_table = chicks.pivot_table(index='Age', columns='Diet', values='Weight', aggfunc='mean', observed=True)
# a table: one row per age, one column per diet, the cells are the average weights
print(mean_table.round(1))
# show it rounded to 1 decimal

**What the charts show:**
- **All chicks start at the same weight**: about **41 grams** on day 0 in every diet (40.7–41.4). So any difference later comes from the diet, not from the starting weight.
- Weight rises steadily with age in every diet, and the lines **spread apart over time**: the gap between diets grows bigger every week.
- **On day 21 the average weights are: Diet 1 ≈ 178 g, Diet 2 ≈ 215 g, Diet 3 ≈ 270 g, Diet 4 ≈ 239 g.** Diet 3 is the highest and Diet 1 the lowest.
- **Diet 4** gives the fastest start (highest average up to about day 12) but **Diet 3 overtakes it from day 14** and ends far ahead.
- Inside each diet the chicks differ a lot (the first chart): on day 21 some Diet 2 chicks are below 100 g and others above 300 g. That big spread inside the groups is why we need a statistical test before we say the diets differ.

---
# Task 3: Statistical Testing

**Weight gain** = weight on day 21 − weight on day 0 (birth). We compute it for the chicks that were weighed on both days, then compare the 4 diets with a one-way **ANOVA** (`f_oneway`).

In [ ]:
start = chicks[chicks['Age'] == 0].set_index('Chick')['Weight']
# the birth weight of every chick (day 0)
end = chicks[chicks['Age'] == 21].set_index('Chick')['Weight']
# the weight on day 21 (only the chicks still in the study)
gain = pd.DataFrame({'start': start, 'end': end}).dropna()
# put both in one table and keep only chicks with both values
gain['gain'] = gain['end'] - gain['start']
# weight gained in 21 days (grams)
gain['Diet'] = per_chick.loc[gain.index, 'diet']
# add the diet of each chick
print('Chicks with a day-21 weight:', len(gain))
# how many chicks we compare

In [ ]:
summary = gain.groupby('Diet', observed=True)['gain'].agg(
    n='size',
    mean=lambda g: stats.tmean(g),
    median='median',
    std=lambda g: stats.tstd(g),
    skew=lambda g: stats.skew(g))
# for each diet: number of chicks, mean (tmean), median, standard deviation (tstd) and skewness of the gain
print(summary.round(2))
# show the table

In [ ]:
groups = [gain.loc[gain['Diet'] == d, 'gain'] for d in [1, 2, 3, 4]]
# a list with the weight gains of each diet
f_value, p_value = stats.f_oneway(*groups)
# one-way ANOVA; H0 = all 4 diets give the same average gain (* passes the 4 groups one by one)
print('F-value:', round(f_value, 3))
# variation between diets / variation inside diets
print('P-value:', p_value)
# the chance of such an F-value if the diets were all the same
print('Significant at 0.05:', p_value < 0.05)
# True = at least one diet differs

In [ ]:
levene_stat, levene_p = stats.levene(*groups)
# Levene test: do the 4 diets have a similar spread? (ANOVA assumes they do)
print('Levene p-value:', round(levene_p, 4))
# p > 0.05 = the spreads are similar enough
kw_stat, kw_p = stats.kruskal(*groups)
# Kruskal-Wallis: an ANOVA on ranks, a check that does not need normal data
print('Kruskal-Wallis p-value:', round(kw_p, 4))
# if this agrees with ANOVA, the result is robust

**Which diets differ?** A T-test (`ttest_ind`) for every pair of diets, with the critical value from `t.ppf`.

In [ ]:
from itertools import combinations
# combinations: makes all the pairs from a list
rows = []
# here we collect one result row per pair
for a, b in combinations([1, 2, 3, 4], 2):
    # the 6 pairs: (1,2), (1,3), (1,4), (2,3), (2,4), (3,4)
    ga, gb = groups[a - 1], groups[b - 1]
    # the gains of the two diets (list positions start at 0)
    t_stat, p = stats.ttest_ind(ga, gb)
    # T-test for the pair
    df_t = len(ga) + len(gb) - 2
    # degrees of freedom = n1 + n2 - 2
    crit = stats.t.ppf(1 - 0.05 / 2, df_t)
    # critical value for alpha = 0.05
    rows.append({'pair': f'Diet {a} vs {b}', 'mean diff': ga.mean() - gb.mean(), 't': t_stat, 'critical': crit, 'p': p, 'significant': abs(t_stat) > crit})
    # save the results of this pair
pairs = pd.DataFrame(rows)
# turn the list into a table
pairs['significant (Bonferroni 0.05/6)'] = pairs['p'] < 0.05 / 6
# stricter check, because we did 6 tests
print(pairs.round(4).to_string(index=False))
# show the table without the row numbers

In [ ]:
plt.figure(figsize=(8, 5))
# a new chart
sns.boxplot(data=gain, x='Diet', y='gain')
# one box per diet
sns.stripplot(data=gain, x='Diet', y='gain', color='black', alpha=0.6)
# every chick as a black dot on top of the boxes
plt.title(f'Weight gain day 0 → 21 by diet (ANOVA F = {f_value:.2f}, p = {p_value:.4f})')
# results in the title
plt.ylabel('Weight gain (grams)')
# label of the y axis
plt.show()
# show the chart

**Interpretation:**
- **45 chicks** have both a birth weight and a day-21 weight. Their **average weight gain** is: **Diet 1 ≈ 136 g, Diet 2 ≈ 174 g, Diet 3 ≈ 230 g, Diet 4 ≈ 198 g** (the medians tell the same story). The skewness is small (−0.36 to 0.76), so the gains are roughly symmetric.
- **ANOVA:** **F = 4.698**, **p = 0.0066**. The P-value is below 0.05, so we **reject H0**: the diets do **not** all give the same weight gain. At least one diet differs.
- **Checks:** the Levene test (p = 0.335) says the spreads of the 4 groups are similar enough, which ANOVA assumes. The Kruskal–Wallis test, which does not need normal data, also gives a significant result (p = 0.013). So the conclusion is robust.
- **Which pairs differ (T-tests):**
  - **Diet 3 vs Diet 1:** +93 g, **t = 3.63 > 2.06 (critical value), p = 0.0013**. This is significant even with the strict Bonferroni limit (0.05 / 6 ≈ 0.0083).
  - **Diet 4 vs Diet 1:** +61 g, p = 0.012. This is significant at 0.05 but not with the strict limit.
  - **All other pairs** (1 vs 2, 2 vs 3, 2 vs 4, 3 vs 4) are **not significant** (p from 0.12 to 0.44). With only 9–10 chicks per group and a large spread, differences of 24–56 g cannot be proven.
- **In short:** Diet 1 clearly gives the smallest gain; Diet 3 gives the largest, and its advantage over Diet 1 is solid.

---
# Task 4: Growth Analysis

**Growth rate of each chick** = the slope of a straight line through its weights (`linregress` of Weight on Age), in **grams per day**. This uses every measurement, so it also works for the chicks that left early (we only skip a chick with fewer than 3 measurements).

In [ ]:
rates = []
# here we collect one row per chick
for chick_id, g in chicks.groupby('Chick'):
    # go over the chicks one by one; g = the rows of this chick
    if len(g) < 3:
        # a line through 1 or 2 points says almost nothing
        continue
        # skip this chick
    fit = stats.linregress(g['Age'], g['Weight'])
    # the best straight line: Weight = intercept + slope * Age
    rates.append({'Chick': chick_id, 'Diet': g['Diet'].iloc[0], 'rate': fit.slope, 'r2': fit.rvalue**2})
    # save the slope (grams per day) and how well the line fits
rates = pd.DataFrame(rates)
# turn the list into a table
print('Chicks analysed:', len(rates))
# how many chicks got a growth rate
print(rates.groupby('Diet', observed=True)[['rate', 'r2']].agg(['mean', 'std']).round(2))
# average growth rate and fit (R²) for each diet

In [ ]:
rate_groups = [rates.loc[rates['Diet'] == d, 'rate'] for d in [1, 2, 3, 4]]
# the growth rates of each diet
f_rate, p_rate = stats.f_oneway(*rate_groups)
# ANOVA on the growth rates
print('ANOVA on growth rates: F =', round(f_rate, 3), '| p =', p_rate)
# show the result

In [ ]:
daily = chicks.sort_values(['Chick', 'Age']).copy()
# sort by chick and age, so the rows of each chick are in time order
daily['grams_per_day'] = daily.groupby('Chick')['Weight'].diff() / daily.groupby('Chick')['Age'].diff()
# the weight change since the previous weighing, divided by the days between them
period_rate = daily.groupby(['Diet', 'Age'], observed=True)['grams_per_day'].mean().unstack(0)
# average daily growth for each diet at each age
plt.figure(figsize=(10, 5))
# a new chart
for d in period_rate.columns:
    # one line per diet
    plt.plot(period_rate.index, period_rate[d], marker='o', label=f'Diet {d}')
    # the daily growth over age
plt.title('Average daily weight gain by age')
# chart title
plt.xlabel('Age (days)')
# label of the x axis
plt.ylabel('Grams per day')
# label of the y axis
plt.legend()
# show the labels box
plt.show()
# show the chart
daily['stage'] = np.where(daily['Age'] <= 10, 'days 0-10', 'days 11-21')
# label each weighing as the first or the second half of the study
print(daily.groupby(['stage', 'Diet'], observed=True)['grams_per_day'].mean().unstack(1).round(2))
# the average daily gain of each diet in each half

In [ ]:
corr, p_corr = stats.pearsonr(chicks['Age'], chicks['Weight'])
# Pearson correlation between age and weight (all chicks together)
print('Pearson r (age, weight):', round(corr, 3), '| p =', f'{p_corr:.2e}')
# a value close to 1 = weight rises almost in a straight line with age

**Interpretation:**
- **Growth rate (slope of each chick's line):** on average **Diet 1 ≈ 6.3 g/day, Diet 2 ≈ 8.6 g/day, Diet 3 ≈ 11.4 g/day, Diet 4 ≈ 9.5 g/day** (49 chicks; only chick 18, with 2 weighings, is skipped). The straight lines fit well (average R² 0.93–0.98), so "grams per day" is a fair summary of each chick.
- **ANOVA on the growth rates:** **F = 5.593**, **p = 0.0024**, which is even clearer than the weight-gain test. It is significant because this test uses every chick, including the ones that left early.
- **Diet 4** has the **most consistent** growth (smallest std, 2.2 g/day, and the best fit, R² 0.98): its chicks grow at a similar speed. Diets 1 and 2 have the most variable chicks.
- **Speed over time:** in **days 0–10** Diet 4 is fastest (8.5 g/day vs 7.6 for Diet 3). In **days 11–21** Diet 3 jumps to **13.7 g/day**, while Diet 4 stays around 9.3. This is why Diet 3 overtakes Diet 4 in the second half. All diets grow faster in the second half than in the first. The daily chart is jumpy (for example, a dip around day 14 in every diet, and the last step is only 1 day long), so the two-half averages are easier to read.
- **Age and weight** have a Pearson correlation of **r = 0.837** (p ≈ 5e-153): age explains most of a chick's weight, and the diet adds the differences on top.

---
# Task 5: Report Findings

## Report: How diet affects chick growth

**Data.** 578 weighings of **50 chicks** from birth (day 0) to day 21, on 4 diets (20 chicks on Diet 1, 10 on each of the others). There are no empty cells, but **5 chicks left the study early, 4 of them on Diet 1**. All groups start at the same weight (≈ 41 g), so the comparison is fair.

**Main results (supported by SciPy statistics):**

| Diet | Avg weight day 21 | Avg gain day 0→21 | Avg growth rate |
|---|---|---|---|
| 1 | 178 g | 136 g | 6.3 g/day |
| 2 | 215 g | 174 g | 8.6 g/day |
| 3 | **270 g** | **230 g** | **11.4 g/day** |
| 4 | 239 g | 198 g | 9.5 g/day |

1. **The diets have a significant effect on growth.** ANOVA on weight gain: F = 4.70, p = 0.0066. ANOVA on growth rate: F = 5.59, p = 0.0024. The Kruskal–Wallis test confirms it (p = 0.013).
2. **Diet 3 gives the most growth.** Its chicks gained about **94 g more than Diet 1 (≈ 70% more)**, a difference that is significant even with a strict multiple-test correction (p = 0.0013).
3. **Diet 1 gives the least growth** and also had the most chicks dropping out.
4. **Diet 4 is the fast, steady starter.** It is the best diet in the first 10 days and has the most uniform chicks, but it slows down later and ends second.
5. **Diet 2 is in the middle**, with the most variation between chicks. Its differences from the other diets are not statistically significant.

**Limitations.** The groups are small (9–10 chicks on Diets 2–4), so medium differences (such as Diet 3 vs Diet 4, 32 g) cannot be proven. Dropped-out chicks are missing from the day-21 comparison (the growth-rate analysis partly fixes this). We do not know what is in each diet or the reason the chicks dropped out.

**Practical implications.**
- For the **heaviest birds at 3 weeks**, choose **Diet 3**.
- If **early growth and a uniform flock** matter (for example, birds sold earlier), **Diet 4** is a strong choice; a farm could also test Diet 4 early and then Diet 3.
- **Avoid Diet 1**: it gives the slowest growth and the most losses.
- Before a large change, repeat the trial with more chicks per diet, so that the smaller differences can be confirmed and the cost of each diet can be compared against the extra weight.